## 1. Loading the dataset into a pandas dataframe and seeing its dimensions


In [2]:
import pandas as pd

df_ais = pd.read_csv('../data/raw/aisdk-2024-08-07.csv')
print(df_ais.shape)

(28203137, 26)


The dataset contains 28,203,137 AIS records and 26 columns representing 24 hours of vessel activity 

## 2. Inspecting the dataset column structure

In [3]:
print(df_ais.columns)

Index(['# Timestamp', 'Type of mobile', 'MMSI', 'Latitude', 'Longitude',
       'Navigational status', 'ROT', 'SOG', 'COG', 'Heading', 'IMO',
       'Callsign', 'Name', 'Ship type', 'Cargo type', 'Width', 'Length',
       'Type of position fixing device', 'Draught', 'Destination', 'ETA',
       'Data source type', 'A', 'B', 'C', 'D'],
      dtype='object')


The dataset contains 26 columns. The ones most relevant to this project are: 
- 'MMSI' -> unique vessel identifier
- 'Latitude' and 'Longitude' -> vessel position
- 'SOG' -> speed over ground in knots(kn)
- 'Ship type' -> vessel category
- 'Destination' -> reported destination(where available)
- 'ETA' -> reported ETA(where available)
- '# Timestamp' -> timestamp from the AIS basestation
- 'Type of mobile' -> type of target this message is received from (class A AIS Vessel, Class B AIS vessel, etc)
- 'Navigational status' -> navigational status from AIS message (where available), e.g.: 'Engaged in fishing', 'Under way using engine'
- 'Name' -> vessel name 

## 3. Exploring vessel records and target vessel types
Since the dataset has multiple million observations we will find:
- unique vessel MMSI's
- observations per unique MMSI
- unique vessel types(we target cargo, tanker and passenger)
- destination and ETA availability across unique MMSI's with our target type

In [4]:
df_vessels = df_ais['MMSI'].nunique()
print(df_vessels)

df_obs = df_ais.groupby('MMSI').size()
print(df_obs)

vessel_obs = df_ais[df_ais['Type of mobile'] == 'Class A'].groupby('MMSI').size()
print(vessel_obs)

9102
MMSI
148           6902
3638         50431
2112061        880
2182807        924
2182809        874
             ...  
992761024       14
992761025       14
992761026        3
992761086        8
999999991     1606
Length: 9102, dtype: int64
MMSI
148           6902
3638         50431
2190045      30514
102178792        1
111219504    30338
             ...  
660610720        1
735059037     7715
797363629        1
887222300      297
912191373     1237
Length: 3484, dtype: int64


There are 9102 unique MMSI's in the dataset.  
Some MMSI's, such as 148 and 3638, appear unusually short or non-standard. Since the analysis focuses on commercial cargo, tanker, and passenger vessels, Type of mobile = Class A is relevant, but unusual MMSI values still remain even within Class A records.

In [5]:
df_types = df_ais.groupby('Ship type').size()
print(df_types)

Ship type
Anti-pollution             61388
Cargo                    5628117
Diving                     27893
Dredging                  580502
Fishing                  4225151
HSC                       868557
Law enforcement           134255
Medical                     8468
Military                  269527
Not party to conflict      18144
Other                     938883
Passenger                3566724
Pilot                     724937
Pleasure                 1425235
Port tender                81058
Reserved                  132764
SAR                       610755
Sailing                  2497168
Spare 1                    21021
Spare 2                     8237
Tanker                   2647622
Towing                     48080
Towing long/wide           60103
Tug                       875809
Undefined                2742397
WIG                          342
dtype: int64


The raw dataset contains:
- 5,628,117 cargo observations
- 2,647,622 tanker observations
- 3,566,724 passenger observations

In [6]:
unique_mmsi_by_type = df_ais.groupby('Ship type')['MMSI'].nunique()
print(unique_mmsi_by_type)

Ship type
Anti-pollution              8
Cargo                     694
Diving                     19
Dredging                   81
Fishing                   643
HSC                        73
Law enforcement            44
Medical                     1
Military                   80
Not party to conflict       4
Other                     233
Passenger                 435
Pilot                      80
Pleasure                 2123
Port tender                26
Reserved                   19
SAR                       168
Sailing                  2662
Spare 1                     4
Spare 2                     5
Tanker                    260
Towing                     17
Towing long/wide            6
Tug                       168
Undefined                8284
WIG                         2
Name: MMSI, dtype: int64


Among the target vessel types the dataset contains:
- 694 cargo MMSI's
- 260 tanker MMSI's
- 435 passenger MMSI's  
- Keep in mind each MMSI does not necessarily represent an actual vessel

In [7]:
df_target = df_ais[df_ais['Ship type'].isin(['Cargo', 'Tanker', 'Passenger'])]
eta_availability = df_target.groupby('Ship type')['ETA'].count()
dest_availability = df_target.groupby('Ship type')['Destination'].count()
print(eta_availability)
print(dest_availability)

Ship type
Cargo        5522543
Passenger    2793477
Tanker       2627667
Name: ETA, dtype: int64
Ship type
Cargo        5628117
Passenger    3566724
Tanker       2647622
Name: Destination, dtype: int64


ETA availability:
- 5,522,543 cargo observations
- 2,793,477 passenger observations
- 2,627,667 tanker observations

Destination availability:
- 5,628,117 cargo observations
- 3,566,724 passenger observations
- 2,647,622 tanker observations  
- We've established that all target observations have a non-null destination field

Sidenote: non-null observations such as 'Unknown' do not necessarily count as valid ones

## 4. Absence and field quality of key values 
We will investigate:  
- MMSI  
- Latitude  
- Longitude
- SOG  
- Navigational Status
- Most common destinations  
- ETA format and validity  
- Navigational Status validity  
- SOG values and suspicious speeds

In [8]:
mmsi_count = df_ais['MMSI'].notnull().sum()
print(mmsi_count)

lat_count = df_ais['Latitude'].notnull().sum()
lon_count = df_ais['Longitude'].notnull().sum()
print(lat_count)
print(lon_count)

sog_count = df_ais['SOG'].notnull().sum()
print(sog_count)

nav_status_count = df_ais['Navigational status'].notnull().sum()
print(nav_status_count)

28203137
28203137
28203137
25965988
28203137


MMSI, Latitude, Longitude, and Navigational Status are complete across the dataset. SOG contains missing values(around 2.2 million rows) and will require handling during the cleaning stage

In [9]:
df_dest = df_target['Destination'].value_counts(ascending=False).head(30)
print(df_dest)

Destination
Unknown         362278
SEGOT           194299
BEANR           151872
DK SKA          139244
PLSZZ           136458
PLGDN           128656
FOR ORDERS      119098
PLGDY           113026
DEHAM           105048
DE RSK           83918
UST LUGA         79883
DEHAM CTB        79412
SEMMA<>DETRV     76516
SE GOT           75720
PLGDND>DEBRV     68716
DKAAR            68447
DKCPH            68142
LVLPX            67101
CNYTN            67012
SKAGEN           66915
DERSK            65466
LTKLJ            65071
PLGDN>DEBRV      63791
PL GDY           63355
PLGDN>DKSKA      61077
SE TRG           59692
EGPSD            59125
EG PSD           59084
ESBJERG          58817
SEKAN            57981
Name: count, dtype: int64


Destination values and their format vary. Many use UN/LOCODE-style codes, while others use free-text port names, route strings, or non-specific values such as Unknown and FOR ORDERS. Destination standardization will be required before port-coordinate mapping. There are 362,278 observations with the value 'Unknown'. These observations do not necessarily represent a major issue for the analysis since they could come from a small numbers of really 'chatty' vessels with 'Unknown' values. We will need to check how many unique MMSI's each destination has

In [10]:
df_dest_unique_mmsi = df_target.groupby('Destination')['MMSI'].nunique().sort_values(ascending=False).head(30)
print(df_dest_unique_mmsi)

Destination
Unknown          153
DEHAM             22
SE GOT            17
FOR ORDERS        17
SEGOT             16
BEANR             14
DERSK             14
KIEL              12
PLGDN             12
ESBJERG           11
DK SKA            10
DKCPH             10
LTKLJ              9
DKAAR              9
PL GDN             9
NLRTM              9
SASSNITZ           8
NOK                8
DKSKA              8
PLGDY              8
FOR ORDER          8
SEKAN              8
SKAGEN             7
KLAIPEDA           7
STYRSOBOLAGET      7
GOTEBORG           6
RUKGD              6
DE RSK             6
PLSZZ              6
DEBRV              6
Name: MMSI, dtype: int64


After further investigation 'Unknown' dominates with 153 unique MMSI's. Furthermore destination values are highly inconsistent and each destination appears under multiple names e.g. SE GOT / SEGOT,
DK SKA / DKSKA / SKAGEN,
FOR ORDERS / FOR ORDER,
PLGDN / PL GDN,
DERSK / DE RSK

In [11]:
df_eta = df_target['ETA'].dropna().head(30)
print(df_eta)

28      07/08/2024 04:00:00
29      07/08/2024 04:00:00
30      07/08/2024 04:00:00
35      07/08/2024 04:00:00
39      07/08/2024 04:00:00
380     07/08/2024 14:00:00
456     09/05/2025 07:00:00
483     09/05/2025 07:00:00
935     07/08/2024 14:00:00
937     07/08/2024 14:00:00
1002    08/08/2024 20:00:00
1005    08/08/2024 20:00:00
1006    08/08/2024 20:00:00
1009    08/08/2024 20:00:00
1218    08/08/2024 20:00:00
1219    08/08/2024 20:00:00
1222    08/08/2024 20:00:00
1225    08/08/2024 20:00:00
1226    08/08/2024 20:00:00
1279    07/07/2025 05:00:00
1748    08/08/2024 20:00:00
1755    08/08/2024 20:00:00
1902    06/08/2024 21:30:00
1908    06/08/2024 21:30:00
1976    08/08/2024 04:00:00
1982    08/08/2024 04:00:00
1983    08/08/2024 04:00:00
1984    08/08/2024 04:00:00
1985    08/08/2024 04:00:00
1986    08/08/2024 04:00:00
Name: ETA, dtype: object


The ETA field contains full datetime values and is often repeated across multiple AIS observations for the same vessel.
Because this project measures vessel progress during the 24-hour tracking period, the ETA does not need to fall within or close to the observed 24-hour window. Vessels may be at very different stages of their voyages

In [12]:
df_nav_status = df_target['Navigational status'].value_counts()
print(df_nav_status)

Navigational status
Under way using engine                 10798493
Constrained by her draught               280034
Unknown value                            237170
At anchor                                213377
Moored                                   203910
Under way sailing                         55119
Restricted maneuverability                47954
Not under command                          5661
Reserved for future amendment [HSC]         670
Engaged in fishing                           59
Aground                                      16
Name: count, dtype: int64


Most target vessel observations are classified as 'Under way using engine' with a precise count of 10.8 million observations.
Other relevant states include:
- 'Constrained by her draught'-  280,034 observations
- 'At anchor'- 213,377 observations
- 'Moored'-  203,910 observations
- 'Restricted maneuverability'-  47,954 observations

In [13]:
pd.set_option('display.float_format', lambda x: f"{x:.2f}")
print(df_target['SOG'].describe())

count   11823317.00
mean           9.95
std            5.60
min            0.00
25%            7.70
50%           10.90
75%           13.60
max           99.70
Name: SOG, dtype: float64


Speed values are as expected for commercial vessels with the maximum of 99.7 knots being invalid/suspicious